## GNN4ID Heterogeneous Graph Model

In this notebook, we provide instructions for using our developed heterogeneous graph models. We have created two different architectures:

1. **Model without Edge Attributes**: In this model, edges provide only the connection information between nodes. This means the model focuses solely on the structural relationships within the graph.
2. **Model with Edge Attributes**: In this model, edges have their own attributes/features in addition to providing connection information between nodes. This allows the model to leverage additional information carried by the edges, potentially improving its performance and insights.

In [5]:
from Utility.Functions import *
from Utility.Model import *
from Utility.Training import *
from Utility.Additional_Features import *
from torch_geometric.loader import DataLoader
from tqdm import tqdm

### Reading Graph Objects

**dir**: Where grapgh data is stored in processed folder.
    data directory will have two folders inside: raw and processed.
    graph objects will be stored in this processed folder

In [6]:
Dict_x = {'Benign': 0 , 
          'WebBased': 1, 
          'Spoofing': 2,
          'Recon' : 3,
          'Mirai' : 4,
          'Dos' : 5,
          'DDos' : 6,
          'BruteForce': 7
         }

dir = "F:/CIC_IOT/Extracted_Flow_Features/" ## Directory where graph data will be stored
Files =glob.glob("F:/CIC_IOT/Extracted_Flow_Features/train/*.csv") ## Directory where CSV files(Extracted Flow-level and packet-level inforamtion) is stored

In [7]:
data_Hetero = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files, skip_processing=True, test=False, single_file=True)

In [ ]:
data_Hetero

### Initializing the Model

In [9]:
## Arguments for running the model
args = {
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'hidden_size': 64,
    'epochs': 30,
    'weight_decay': 1e-5,
    'lr': 0.01,
    'attn_size': 32,
    'eps': 1.0,
}

In [10]:
## Initializing a Data Instance for Model Initialization
data_model=data_Hetero[0].to(args['device'])

In [11]:
## Paper-faithful HGNN with GATConv + edge attributes (Section 3.1.4).
model = HeteroGNN(data_model, args, aggr="mean").to(args['device'])

## Ablation: same skeleton without edge attributes
# model = HeteroGNN(data_model, args, aggr="mean", use_edge_attr=False).to(args['device'])

## Legacy SAGE-based ablation (NOT paper architecture)
# from Utility.Model import HeteroGNN_SAGE
# model = HeteroGNN_SAGE(data_model, args, aggr="mean").to(args['device'])


### Training Loop


In [23]:
train_loader = DataLoader(data_Hetero, batch_size=64, shuffle=True)

In [ ]:
# Trains the paper-faithful HGNN (with edge attributes).
train(train_loader, model, args, args["device"])


### Testing Loop

In [12]:
data_Hetero = NIDSDataset(root=dir, label_dict=Dict_x, filename=Files, skip_processing=True, test=True, single_file=True)

In [13]:
## For testing the model
test_loader = DataLoader(data_Hetero, batch_size=1, shuffle=False)

In [ ]:
# Evaluate. test_cm() routes through the edge-attr-aware path by default.
acc, prediction, label = test_cm(test_loader, model)


#### Classification Report

In [ ]:
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
print(classification_report(label,prediction))
print('\n')
print('                    Accuracy %',(round(accuracy_score(label,prediction),4)*100))
print('\n')

#### Confusion Matrix

In [ ]:
cm=confusion_matrix(label,prediction, normalize='true') ## Getting Results in Percentage 
plt.figure(figsize=(10, 10))
ax = plt.axes()
sns.heatmap(cm, annot=True, cmap='Blues', fmt='.1%',ax=ax) # fmt= 'd' for just showing the value in int
ax.set_ylabel('True Label') 
ax.set_xlabel('Predicted label')
labels=['Benign','WebBased','Spoofing','Recon','Mirai','Dos','DDos','BruteForce']
ax.xaxis.set_ticklabels(labels); ax.yaxis.set_ticklabels(labels)
plt.show()


#### Saving/Loading Model

In [ ]:
torch.save(model, '/scratch/user/yasir.ali/GNN_Project/Saved_Model/GNN4ID_8_Classes/model.pth')
# model = torch.load('/scratch/user/yasir.ali/GNN_Project/Saved_Model/GNN4ID_8_Classes/model.pth')

### Explainability: Integrated Gradient + Generative LLM

Following paper Section 3.1.5 (Integrated Gradient Explainer, eq. 10) and
Section 3.1.6 / Algorithm 3 (Generative Explainer with Llama-3-8B). The IG
step does not need the LLM and is fast; the LLM step is optional and
requires a HuggingFace pipeline.

In [ ]:
from Utility.IG_Explainer import IntegratedGradientExplainer, top_flow_features, top_payload_bytes
from Utility.LLM_Explainer import LLMExplainer, load_llama3_pipeline, DEFAULT_CLASS_NAMES

# Load a single test sample (model already trained above).
test_sample = data_Hetero[0]

# Flow feature names: read them once from the training CSV header. Drop the
# columns NIDSDataset removes before exposing them as flow node features.
import pandas as pd
_train_csv = Files[0]
_drop = {'src_ip','src_port','dst_ip','dst_port','ip_version','bidirectional_bytes',
         'bidirectional_first_seen_ms','bidirectional_last_seen_ms','bidirectional_duration_ms',
         'bidirectional_packets','src2dst_first_seen_ms','src2dst_last_seen_ms',
         'dst2src_first_seen_ms','dst2src_last_seen_ms','id','src_mac','src_oui','dst_mac',
         'dst_oui','vlan_id','tunnel_id','bidirectional_syn_packets','bidirectional_cwr_packets',
         'bidirectional_ece_packets','bidirectional_urg_packets','bidirectional_ack_packets',
         'bidirectional_psh_packets','bidirectional_rst_packets','bidirectional_fin_packets',
         'expiration_id','protocol','Label','udps.payload_data','udps.delta_time',
         'udps.packet_direction','udps.ip_size','udps.transport_size','udps.payload_size',
         'udps.syn','udps.cwr','udps.ece','udps.urg','udps.ack','udps.psh','udps.rst','udps.fin',
         'is_vulnerable_port','is_http_port','is_dns_dst_port','is_dns_src_port','is_vuln_port',
         'is_udp_request','is_tcp_request','is_icmp_request'}
_header = list(pd.read_csv(_train_csv, nrows=0).columns)
FLOW_FEATURE_NAMES = [c for c in _header if c not in _drop]
# After NIDSDataset one-hot encodes expiration_id and protocol, the dummy
# column names are appended. Reproduce them in the same order.
FLOW_FEATURE_NAMES += [f'Exp_{v}' for v in (0, -1)]
FLOW_FEATURE_NAMES += [f'proto_{v}' for v in (1, 2, 6, 17, 58)]
print('Number of flow feature names:', len(FLOW_FEATURE_NAMES))
print('Flow node tensor dim:', test_sample['flow'].x.shape)


In [ ]:
ig = IntegratedGradientExplainer(model, device=args['device'], n_steps=50)
attr = ig.explain(test_sample)
print('Predicted class:', DEFAULT_CLASS_NAMES.get(attr.predicted_class, attr.predicted_class))
print('Top flow features (name, IG attr, actual value):')
for name, a, v in top_flow_features(attr, FLOW_FEATURE_NAMES, top_n=5):
    print(f'  {name:40s}  attr={a:+.4f}  value={v:.4f}')


In [ ]:
# Build the LLM prompt. Pass pipeline=None for a dry run; load Llama-3 to
# actually generate the explanation.
explainer = LLMExplainer()
explanation = explainer.explain(attr, FLOW_FEATURE_NAMES, pipeline=None)
print('=== Flow prompt ===')
print(explanation.flow_prompt)
if explanation.payload_prompt:
    print('\n=== Payload prompt ===')
    print(explanation.payload_prompt)


In [ ]:
# Optional: actually call Llama-3-8B. Requires HuggingFace login and a GPU
# with >=12 GB VRAM (4-bit quantization).
# pipe = load_llama3_pipeline()
# explanation = explainer.explain(attr, FLOW_FEATURE_NAMES, pipeline=pipe)
# print(explanation.text)
